# Example 3: Two species in 1D

This notebook walks through [`eg_3.py`](eg_3.py). See [`README.md`](README.md) for the
full problem statement.

## The problem

Two species, `c1` and `c2`, each diffusing on the rod $(0, 1)$ with their own boundary
conditions, solved **together** in one system:

$$ \frac{\partial c_i}{\partial t} - \nabla\cdot(k\,\nabla c_i) = 0,
\qquad i \in \{1, 2\}. $$

They do not interact here; the point is to learn how to carry **several fields** at
once. Coupling them comes in Example 4.

## What is new

A **mixed function space** that packs both fields into one unknown `u`, the
**split/collapse** tools to move between the combined unknown and the individual
fields, and **boundary conditions applied to sub-spaces**.

## 1. Backends and imports

The same backends as before (`mpi4py` for parallelism, `petsc4py` for solvers), plus
two new tools for a time-dependent run:

- **`tqdm`** for a progress bar over the time steps,
- **`dolfinx.io.VTXWriter`** to write the solution at each step to a `.bp` file you can
  open in ParaView.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc

import dolfinx
import basix
import numpy as np
import tqdm.autonotebook
from dolfinx import fem
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.mesh import create_mesh, locate_entities_boundary, meshtags
import ufl

from dolfinx.io import VTXWriter

## 2. The mesh

Unchanged from Example 1: a uniform 1D interval mesh on $(0, 1)$.

In [ ]:
# 100 points, 99 interval cells, on the domain (0, 1)
indices = np.linspace(0, 1, num=100)
gdim, shape, degree = 1, "interval", 1
domain = ufl.Mesh(basix.ufl.element("Lagrange", shape, degree, shape=(gdim,)))
mesh_points = np.reshape(indices, (len(indices), 1))
indexes = np.arange(mesh_points.shape[0])
cells = np.stack((indexes[:-1], indexes[1:]), axis=-1)
my_mesh = create_mesh(comm=MPI.COMM_WORLD, cells=cells, x=mesh_points, e=domain)
fdim = my_mesh.topology.dim - 1

## 3. A mixed function space

To solve for two fields together we build a **mixed element** from two copies of a
continuous (CG) Lagrange element, then make one function space `V` from it. The single
unknown `u` now holds **both** `c1` and `c2`.

A few helpers let us move between the combined unknown and the individual fields:

- `ufl.split(u)` gives the symbolic pieces `c1, c2` used to write the weak form,
- `V.sub(i)` is the sub-space for field `i` (used for boundary conditions),
- `u.sub(i).collapse()` extracts a standalone function for output,
- the collapse maps tell us which entries of `u.x.array` belong to each field.

In [ ]:
element_CG = basix.ufl.element(
    basix.ElementFamily.P,
    my_mesh.basix_cell(),
    1,
    basix.LagrangeVariant.equispaced,
)
elements = basix.ufl.mixed_element([element_CG, element_CG])

V = fem.functionspace(my_mesh, elements)
u = fem.Function(V)        # holds both fields (current step)
u_n = fem.Function(V)      # previous step

c1, c2 = ufl.split(u)
c1_n, c2_n = ufl.split(u_n)
V1, V2 = V.sub(0), V.sub(1)
v1, v2 = ufl.TestFunction(V)
c1_pp, c2_pp = u.sub(0).collapse(), u.sub(1).collapse()
_, map_c1_to_u = V.sub(0).collapse()
_, map_c2_to_u = V.sub(1).collapse()

## 4. Boundary conditions on sub-spaces

With more than one field we label the boundary explicitly. We tag the left facet `1`
and the right facet `2` with `meshtags`, find the degrees of freedom of each **field**
on those facets with `locate_dofs_topological(V.sub(i), ...)`, and pin them.

Here `c1` runs from `100` to `0` and `c2` from `75` to `0`.

In [ ]:
num_facets = my_mesh.topology.index_map(fdim).size_local
mesh_facet_indices = np.arange(num_facets, dtype=np.int32)
tags_facets = np.full(num_facets, 0, dtype=np.int32)

entities_left = locate_entities_boundary(my_mesh, fdim, lambda x: np.isclose(x[0], 0))
entities_right = locate_entities_boundary(my_mesh, fdim, lambda x: np.isclose(x[0], 1))
tags_facets[entities_left] = 1
tags_facets[entities_right] = 2

facet_meshtags = meshtags(my_mesh, fdim, mesh_facet_indices, tags_facets)
my_mesh.topology.create_connectivity(fdim, my_mesh.topology.dim)

left_facets = facet_meshtags.find(1)
left_dofs_c1 = fem.locate_dofs_topological(V.sub(0), fdim, left_facets)
left_dofs_c2 = fem.locate_dofs_topological(V.sub(1), fdim, left_facets)
right_facets = facet_meshtags.find(2)
right_dofs_c1 = fem.locate_dofs_topological(V.sub(0), fdim, right_facets)
right_dofs_c2 = fem.locate_dofs_topological(V.sub(1), fdim, right_facets)

bc_left_c1 = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(100)), left_dofs_c1, V1)
bc_left_c2 = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(75)), left_dofs_c2, V2)
bc_right_c1 = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(0)), right_dofs_c1, V1)
bc_right_c2 = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(0)), right_dofs_c2, V2)
bcs = [bc_left_c1, bc_left_c2, bc_right_c1, bc_right_c2]

## 5. The variational form

Each field gets its own diffusion term and its own backward-Euler time term, tested
against its own test function (`v1` or `v2`). We **add** them all into one residual
`F`. Because the two fields share no terms, they are effectively two independent
problems solved in one system.

In [ ]:
k = 0.1
dt = 0.1
F = ufl.dot(k * ufl.grad(c1), ufl.grad(v1)) * ufl.dx
F += ufl.dot(k * ufl.grad(c2), ufl.grad(v2)) * ufl.dx
F += ((c1 - c1_n) / dt) * v1 * ufl.dx
F += ((c2 - c2_n) / dt) * v2 * ufl.dx

## 6. The solver

Identical to the earlier examples: a `NonlinearProblem` driven by PETSc's SNES
(Newton) solver, configured with a direct LU solve (MUMPS) at each step. The trailing
loop clears the options out of the global PETSc database so they do not leak into
other solvers. See Example 1 for the line-by-line explanation.

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_stol": np.sqrt(np.finfo(dolfinx.default_real_type).eps) * 1e-2,
    "snes_atol": 1e-10,
    "snes_rtol": 1e-10,
    "snes_max_it": 30,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
solver = NonlinearProblem(
    F,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="festim_solver",
)

# clear the options out of the global PETSc database
snes = solver.solver
prefix = snes.getOptionsPrefix()
opts = PETSc.Options()
for key in petsc_options.keys():
    del opts[f"{prefix}{key}"]

## 7. Solve in a time loop

As in Example 2, but with **two output files**, one per species. At each step we copy
the collapse-mapped entries of `u` into the standalone functions `c1_pp` and `c2_pp`
before writing them.

In [ ]:
writer1 = VTXWriter(MPI.COMM_WORLD, "two_species_c1.bp", c1_pp, "BP5")
writer2 = VTXWriter(MPI.COMM_WORLD, "two_species_c2.bp", c2_pp, "BP5")

final_time = 10
t = 0
progress = tqdm.autonotebook.tqdm(
    desc="Solving two species problem", total=final_time, unit_scale=True
)
while t < final_time:
    solver.solve()
    u_n.x.array[:] = u.x.array

    c1_pp.x.array[:] = u.x.array[map_c1_to_u]
    c2_pp.x.array[:] = u.x.array[map_c2_to_u]

    writer1.write(t)
    writer2.write(t)

    t += dt
    progress.update(dt)

writer1.close()
writer2.close()

## Recap

You learned how to carry several fields in one mixed function space, split and collapse
between the combined unknown and the individual fields, and apply boundary conditions
per sub-space. Open `two_species_c1.bp` and `two_species_c2.bp` in
[ParaView](https://www.paraview.org/) to compare the two profiles.